# 07b — Geometric feature extraction

**Research question:** Đặc trưng hình học mô tả gì và có tách biệt khỏi chu vi mục tiêu không?

Dataset BodyM; baseline Ridge dùng hai silhouette + metadata. Không train lại hoặc đổi model đã chọn.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Module_01_User_and_Body_Profile.md").exists())
for path in [ROOT, ROOT / "scripts"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from IPython.display import display, Markdown
import numpy as np
import matplotlib.pyplot as plt


In [2]:
from run_bodym_baseline import silhouette_features, FEATURE_NAMES, read_csv
path = ROOT / "data/raw/bodym/train"
row = sorted(read_csv(path / "subject_to_photo_map.csv"), key=lambda r: (r["subject_id"], r["photo_id"]))[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, view in zip(axes, ("mask", "mask_left")):
    features = silhouette_features(path / view / (row["photo_id"] + ".png"))
    assert features.shape == (195,) and np.isfinite(features).all()
    for i, name in enumerate(("span", "occupied", "center")):
        ax.plot(np.linspace(0, 1, 64), features[i * 64:(i + 1) * 64], label=name)
    print(view, dict(zip(("aspect_ratio", "area", "fill_ratio"), features[-3:])))
    ax.set_xlabel("Normalized body height (top to bottom)"); ax.set_title(view); ax.legend()
fig.tight_layout(); plt.show()
assert len(FEATURE_NAMES) == 393
assert not {"chest", "waist", "hip"} & set(FEATURE_NAMES)
print("Features: 3 metadata + 2 x (3 profiles x 64 bins + 3 scalars) =", len(FEATURE_NAMES))

mask {'aspect_ratio': np.float64(0.6065375302663438), 'area': np.float64(0.1763916015625), 'fill_ratio': np.float64(0.2913306451612903)}
mask_left {'aspect_ratio': np.float64(0.17375886524822695), 'area': np.float64(0.107818603515625), 'fill_ratio': np.float64(0.6273082386363636)}
Features: 3 metadata + 2 x (3 profiles x 64 bins + 3 scalars) = 393


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18908\154925102.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


## Metrics / error analysis / conclusion

Profile span gồm cả khoảng trống giữa tay/chân; occupied chỉ đếm foreground. Center mô tả bất đối xứng. Đều chuẩn hóa theo chiều cao ảnh đã crop, không phải cm. Metadata chiều cao/cân nặng được cung cấp; chu vi ngực/eo/hông chỉ nằm ở y. Ridge dùng 393 features; CNN học encoder trực tiếp từ ảnh, không dùng bộ feature Ridge này. Xem notebook 08 và 09 cho kết quả train đã có; notebook 10 bổ sung đánh giá Phase 2.